# Severity Ranking Framework

This notebook is the starting point for the severity model. From our original project proposal, the goal here isn't just a simple rank of "this seems like it's more severe" or a model to determine outside factors that may influence severity. Instead, we want to create a model that learns what the indicators are for increasingly severe issues (such as ones more likely to lead to injuries, deaths, car fire, etc) and assigns a numeric urgency score to each complaint so they can then be sorted into bands of how urgent the complaint is. This allows us to then choose the top percentage of these complaints to send to human reviewers so that the most important issues are being addressed first since, realistically, a human reviewer can only go through so many complaints in a given time (the reviewer "budget").

It starts from `data/processed/odi_severity_cases.parquet`, which is produced by the preprocessing stage in the current pipeline.

Goal in this pass:
- use `severity_primary_flag` only
- compare simple structured, text, and hybrid notebook baselines
- decide which model creates the most useful review queue on `valid_2025`

Working reviewer budgets:
- top 1%
- top 2%
- top 5%
- top 10%

How to read this notebook:
- `valid_2025` is the decision set for model choice
- `holdout_2026` is reference-only and stays off by default
- the main question is which score creates the most useful review queue at a realistic review budget, not just which score ranks severe complaints highest

Optional output written by this notebook:
- `data/outputs/severity_partner_results.csv`

## 1. Setup

This notebook stays self-contained on top of the current pipeline outputs. We read the severity-ready processed table directly and builds the exploratory modeling steps here to find what works.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
SEVERITY_PATH = PROCESSED_DATA_DIR / 'odi_severity_cases.parquet'
SEVERITY_RESULTS_PATH = OUTPUTS_DIR / 'severity_partner_results.csv'

RANDOM_SEED = 42
sns.set_theme(style='whitegrid')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Severity input:", SEVERITY_PATH)
print("Exploratory results path:", SEVERITY_RESULTS_PATH)

Project root: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics
Severity input: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\processed\odi_severity_cases.parquet
Exploratory results path: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\outputs\severity_partner_results.csv


## 2. Load The Severity Table

We start from the severity-specific processed table. The raw complaint files and the shared cleaning rules have already been handled upstream.

In [2]:
if not SEVERITY_PATH.exists():
    raise FileNotFoundError(f"Run preprocessing first. Missing input file: {SEVERITY_PATH}")

df = pd.read_parquet(SEVERITY_PATH)
df['ldate'] = pd.to_datetime(df['ldate'])

# Build a cleaned text view directly in the notebook for severity exploration
text_series = df['cdescr'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df['cdescr_model_text'] = text_series

print("Loaded from:", SEVERITY_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df[['odino', 'ldate', 'mfr_name', 'maketxt', 'modeltxt', 'yeartxt', 'severity_primary_flag', 'severity_broad_flag']].head())

Loaded from: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\processed\odi_severity_cases.parquet
Rows: 375937
Columns: 67


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,severity_primary_flag,severity_broad_flag
0,10816121,2020-10-05,"GENERAL MOTORS, LLC",PONTIAC,VIBE,2007,False,False
1,11289434,2020-01-02,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2019,False,False
2,11289436,2020-01-02,HYUNDAI MOTOR AMERICA,HYUNDAI,SONATA,2018,True,True
3,11290552,2020-01-02,"CHRYSLER (FCA US, LLC)",DODGE,DURANGO,2011,True,True
4,11292384,2020-01-01,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,ACCORD,2018,False,False


## 3. Fix The Target And Review Budgets

This pass is intentionally scoped to `severity_primary_flag`. `severity_broad_flag` stays in the table for later sensitivity analysis, but it is not part of model choice here. Because severe complaints are uncommon, we will report PR-AUC and recall-style ranking metrics. The main operating point is the top 5% review queue, with top 1%, top 2%, and top 10% used as supporting checks.

In [3]:
TARGET_COL = 'severity_primary_flag'
REVIEW_BUDGETS = [0.01, 0.02, 0.05, 0.10]
REVIEW_BUDGET_LABELS = {
    0.01: 'top_1pct',
    0.02: 'top_2pct',
    0.05: 'top_5pct',
    0.10: 'top_10pct'
}

print("Active target:", TARGET_COL)
print("Primary review budget for model choice: top 5%")

target_summary = pd.DataFrame({
    'target': ['severity_primary_flag', 'severity_broad_flag'],
    'positive_cases': [int(df['severity_primary_flag'].sum()), int(df['severity_broad_flag'].sum())],
    'positive_rate': [float(df['severity_primary_flag'].mean()), float(df['severity_broad_flag'].mean())]
})
display(target_summary)

year_summary = df.assign(year=df['ldate'].dt.year).groupby('year').agg(
    rows=('odino', 'count'),
    primary_rate=('severity_primary_flag', 'mean'),
    broad_rate=('severity_broad_flag', 'mean')
).reset_index()
display(year_summary)

Active target: severity_primary_flag
Primary review budget for model choice: top 5%


,target,positive_cases,positive_rate
0,severity_primary_flag,22706,0.060398
1,severity_broad_flag,34071,0.090630


,year,rows,primary_rate,broad_rate
0,2020,58465,0.066843,0.085077
1,2021,49044,0.071568,0.096607
2,2022,53221,0.065425,0.101708
3,2023,62727,0.061887,0.094026
4,2024,67432,0.049976,0.084826
5,2025,73985,0.052565,0.086288
6,2026,11063,0.06011,0.085329


## 4. Create Time-Aware Splits

The model should learn from older complaints and be tested on later complaints. Do not randomly shuffle rows across years for the final evaluation because that would make the task easier than real future use.

For this notebook:
- train: complaints received through the end of 2024
- validation: complaints received in 2025 and used for model choice
- holdout: complaints received in 2026, frozen from model choice and kept only for an optional final reference read

In [4]:
# Split the severity table by received date so validation and holdout stay future-facing
train_mask = df['ldate'] <= pd.Timestamp('2024-12-31')
valid_mask = (df['ldate'] >= pd.Timestamp('2025-01-01')) & (df['ldate'] <= pd.Timestamp('2025-12-31'))
holdout_mask = df['ldate'] >= pd.Timestamp('2026-01-01')

train_df = df.loc[train_mask].copy()
valid_df = df.loc[valid_mask].copy()
holdout_df = df.loc[holdout_mask].copy()

split_summary = pd.DataFrame({
    'split': ['train_through_2024', 'valid_2025', 'holdout_2026'],
    'rows': [len(train_df), len(valid_df), len(holdout_df)],
    'positive_rate': [train_df[TARGET_COL].mean(), valid_df[TARGET_COL].mean(), holdout_df[TARGET_COL].mean()]
})
display(split_summary)

,split,rows,positive_rate
0,train_through_2024,290889,0.062402
1,valid_2025,73985,0.052565
2,holdout_2026,11063,0.060110


## 5. Metric And Review-Queue Helpers

PR-AUC still matters because the target is rare, but we're focusing on reviewer budget behavior as the main decision surface. The most important question is if reviewers only have the capacity to inspect the top 5% highest scored complaints, which model captures the most true severe cases while keeping that queue useful.

The helper block below is for overall metrics, operating point summaries, urgency band assignment, and light calibration tables.

In [5]:
severity_results = []

# Gets the positive class scores from a model using predict_proba or falling back to decision_function with a sigmoid transform
def get_positive_scores(model, X):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        return proba[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))


# Calculates the recall at the top fraction of cases based on the model's scores
def recall_at_top_fraction(y_true, score, fraction=0.05):
    y_true = np.asarray(y_true).astype(bool)
    score = np.asarray(score)
    n_top = max(1, int(np.ceil(len(score) * fraction)))
    top_idx = np.argsort(score)[-n_top:]
    total_positive = y_true.sum()
    if total_positive == 0:
        return np.nan
    return y_true[top_idx].sum() / total_positive


# Builds one reviewer budget summary row for a score vector
def build_review_budget_row(y_true, score, fraction):
    y_true = np.asarray(y_true).astype(bool)
    score = np.asarray(score)
    n_rows = len(score)
    n_flagged = max(1, int(np.ceil(n_rows * fraction)))
    top_idx = np.argsort(score)[::-1][:n_flagged]
    severe_captured = int(y_true[top_idx].sum())
    total_positive = int(y_true.sum())
    base_rate = float(y_true.mean()) if n_rows else np.nan
    precision = severe_captured / n_flagged if n_flagged else np.nan
    recall = severe_captured / total_positive if total_positive else np.nan
    cutoff = float(score[top_idx].min()) if n_flagged else np.nan
    lift = precision / base_rate if base_rate else np.nan
    return {
        'budget_fraction': fraction,
        'budget_label': REVIEW_BUDGET_LABELS[fraction],
        'flagged_rows': n_flagged,
        'flagged_share': n_flagged / n_rows if n_rows else np.nan,
        'severe_cases_captured': severe_captured,
        'recall_within_flagged_set': recall,
        'precision_within_flagged_set': precision,
        'lift_vs_base_rate': lift,
        'score_cutoff': cutoff
    }


# Builds the operating point table for one model and split
def build_operating_point_table(model_name, split_name, y_true, score, budgets=None):
    budgets = REVIEW_BUDGETS if budgets is None else budgets
    rows = []
    for fraction in budgets:
        row = build_review_budget_row(y_true, score, fraction)
        row['model'] = model_name
        row['split'] = split_name
        rows.append(row)
    return pd.DataFrame(rows)


# Assigns urgency bands by rank so the review buckets stay stable even when scores tie
def assign_urgency_bands(score):
    score_series = pd.Series(score)
    rank_share = score_series.rank(method='first', ascending=False, pct=True)
    band_values = np.select(
        [rank_share <= 0.01, rank_share <= 0.05, rank_share <= 0.10],
        ['high_urgency', 'priority_review', 'watch'],
        default='routine'
    )
    return pd.Series(band_values, index=score_series.index)


# Builds a compact calibration table with score-ranked bins
def build_calibration_table(y_true, score, n_bins=10):
    calibration_df = pd.DataFrame({
        'y_true': np.asarray(y_true).astype(int),
        'score': np.asarray(score)
    })
    if calibration_df.empty:
        return calibration_df
    if calibration_df['score'].nunique() < 2:
        return pd.DataFrame({
            'bin': ['all_scores'],
            'rows': [len(calibration_df)],
            'avg_score': [float(calibration_df['score'].mean())],
            'observed_rate': [float(calibration_df['y_true'].mean())],
            'gap': [float(calibration_df['y_true'].mean() - calibration_df['score'].mean())],
            'min_score': [float(calibration_df['score'].min())],
            'max_score': [float(calibration_df['score'].max())]
        })
    ranked_score = calibration_df['score'].rank(method='first')
    calibration_df['bin'] = pd.qcut(
        ranked_score,
        q=min(n_bins, len(calibration_df)),
        duplicates='drop'
    )
    calibration_table = calibration_df.groupby('bin', observed=False).agg(
        rows=('y_true', 'size'),
        avg_score=('score', 'mean'),
        observed_rate=('y_true', 'mean'),
        min_score=('score', 'min'),
        max_score=('score', 'max')
    ).reset_index()
    calibration_table['gap'] = calibration_table['observed_rate'] - calibration_table['avg_score']
    calibration_table['bin'] = calibration_table['bin'].astype(str)
    return calibration_table


# Chooses the urgency rule using the notebook decision policy
def choose_urgency_model(selection_df, recall_tolerance=0.01):
    selection_df = selection_df.copy()

    best_recall = selection_df['recall_top_5pct'].max()
    recall_pool = selection_df[selection_df['recall_top_5pct'] >= best_recall - recall_tolerance].copy()

    best_precision = recall_pool['precision_top_5pct'].max()
    precision_pool = recall_pool[np.isclose(recall_pool['precision_top_5pct'], best_precision)]

    best_pr_auc = precision_pool['pr_auc'].max()
    pr_pool = precision_pool[np.isclose(precision_pool['pr_auc'], best_pr_auc)].copy()

    best_brier = pr_pool['brier_score'].min()
    winner = pr_pool[np.isclose(pr_pool['brier_score'], best_brier)].sort_values('model').iloc[0]

    selection_df['within_0_01_recall_of_best_top_5pct'] = (
        selection_df['recall_top_5pct'] >= best_recall - recall_tolerance
    )
    selection_df = selection_df.sort_values(
        ['recall_top_5pct', 'precision_top_5pct', 'pr_auc', 'brier_score', 'model'],
        ascending=[False, False, False, True, True]
    ).reset_index(drop=True)
    return selection_df, winner


# Builds a linear model with stochastic gradient descent and early stopping for fast iteration
def build_linear_sgd_model():
    return SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1e-5,
        max_iter=100,
        tol=1e-3,
        class_weight='balanced',
        random_state=RANDOM_SEED,
        early_stopping=True,
        validation_fraction=0.05,
        n_iter_no_change=5
    )


# Keeps one row per model and split so rerunning a notebook cell refreshes the result instead of duplicating it
def upsert_result(row):
    for idx, existing_row in enumerate(severity_results):
        if existing_row['model'] == row['model'] and existing_row['split'] == row['split']:
            severity_results[idx] = row
            return row
    severity_results.append(row)
    return row


# Evaluates the full set of metrics for a given model and dataset, then updates the severity_results list
def evaluate_scores(model_name, split_name, y_true, score, threshold=0.5):
    pred = score >= threshold
    row = {
        'model': model_name,
        'split': split_name,
        'rows': len(y_true),
        'positive_rate': float(np.mean(y_true)),
        'pr_auc': float(average_precision_score(y_true, score)),
        'roc_auc': float(roc_auc_score(y_true, score)),
        'f1_at_0_5': float(f1_score(y_true, pred, zero_division=0)),
        'precision_at_0_5': float(precision_score(y_true, pred, zero_division=0)),
        'recall_at_0_5': float(recall_score(y_true, pred, zero_division=0)),
        'recall_top_1pct': float(recall_at_top_fraction(y_true, score, 0.01)),
        'recall_top_2pct': float(recall_at_top_fraction(y_true, score, 0.02)),
        'recall_top_5pct': float(recall_at_top_fraction(y_true, score, 0.05)),
        'recall_top_10pct': float(recall_at_top_fraction(y_true, score, 0.10)),
        'brier_score': float(brier_score_loss(y_true, score))
    }
    return upsert_result(row)


# Displays the current severity_results as a sorted DataFrame
def show_results(display_table=True):
    results_df = pd.DataFrame(severity_results)
    if results_df.empty:
        if display_table:
            display(results_df)
        return results_df
    results_df = results_df.sort_values(['split', 'pr_auc'], ascending=[True, False]).reset_index(drop=True)
    if display_table:
        display(results_df)
    return results_df

## 6. Naive Baseline

A baseline is a simple model we must beat. If our real models cannot beat it, they are not useful.

This one uses a simple dummy model that predicts the same probability for all cases based on the overall positive rate in the training data. We will use the DummyClassifier from scikit-learn with the 'prior' strategy, which means it will predict the class probabilities based on the distribution of classes in the training data (i.e. for our binary example, if 1% of the target variable is True then DummyClassifier will use predict_proba to match that distribution). After fitting the dummy model, we will get the predicted probabilities for the validation set and evaluate the metrics using our evaluate_scores function.

In [6]:
dummy_model = DummyClassifier(strategy='prior', random_state=RANDOM_SEED)
dummy_model.fit(train_df[['odino']], train_df[TARGET_COL])

dummy_valid_score = dummy_model.predict_proba(valid_df[['odino']])[:, 1]
dummy_scores = evaluate_scores('dummy_prior', 'valid_2025', valid_df[TARGET_COL].to_numpy(), dummy_valid_score)
results_df = show_results()

,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_1pct,recall_top_2pct,recall_top_5pct,recall_top_10pct,brier_score
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.5,0.0,0.0,0.0,0.012085,0.022114,0.05837,0.119054,0.049898


## 7. Structured Baseline

This model uses the leakage-safe structured core: vehicle identity, complaint context, timing, and missingness fields. It intentionally excludes `injured`, `deaths`, `fire`, `crash`, `medical_attn`, `vehicles_towed_yn`, and `datea` because those fields define or are directly related to the severity target.

It also leaves out `fuel_sys` and `trans_type` because those decode fields are almost entirely missing in the severity table and would mostly act like fragile data-availability proxies rather than stable complaint features.

In [7]:
# Derive complaint timing fields up front so the structured core can use them safely
for frame in [train_df, valid_df, holdout_df]:
    frame['complaint_year'] = frame['ldate'].dt.year.astype('int64')
    frame['complaint_month'] = frame['ldate'].dt.month.astype('int64')

# Leakage-safe structured core from the cleaning policy
structured_cat_features = [
    'mfr_name',
    'maketxt',
    'modeltxt',
    'state',
    'cmpl_type',
    'drive_train',
    'fuel_type',
    'police_rpt_yn',
    'repaired_yn',
    'orig_owner_yn'
]

structured_num_features = [
    'yeartxt',
    'miles',
    'veh_speed',
    'lag_days_safe',
    'miles_missing_flag',
    'veh_speed_missing_flag',
    'faildate_trusted_flag',
    'faildate_untrusted_flag',
    'component_count',
    'row_count',
    'complaint_year',
    'complaint_month'
]

structured_cat_features = [c for c in structured_cat_features if c in df.columns]
structured_num_features = [c for c in structured_num_features if c in df.columns]

def prepare_structured_features(source_df):
    X = source_df[structured_cat_features + structured_num_features].copy()
    for col in structured_cat_features:
        X[col] = X[col].astype('string').fillna('missing').astype(str)
    for col in structured_num_features:
        X[col] = pd.to_numeric(X[col], errors='coerce').astype('float64')
    return X

X_train_structured = prepare_structured_features(train_df)
X_valid_structured = prepare_structured_features(valid_df)
X_holdout_structured = prepare_structured_features(holdout_df)

structured_preprocess = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50))
        ]), structured_cat_features),
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), structured_num_features)
    ],
    remainder='drop'
)

X_train_structured_matrix = structured_preprocess.fit_transform(X_train_structured)
X_valid_structured_matrix = structured_preprocess.transform(X_valid_structured)
X_holdout_structured_matrix = structured_preprocess.transform(X_holdout_structured)

structured_model = build_linear_sgd_model()
structured_model.fit(X_train_structured_matrix, train_df[TARGET_COL])
structured_valid_score = get_positive_scores(structured_model, X_valid_structured_matrix)
evaluate_scores('structured_core_sgd', 'valid_2025', valid_df[TARGET_COL].to_numpy(), structured_valid_score)
results_df = show_results()

,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_1pct,recall_top_2pct,recall_top_5pct,recall_top_10pct,brier_score
0,structured_core_sgd,valid_2025,73985,0.052565,0.444691,0.726783,0.344697,0.272388,0.469272,0.173052,0.342247,0.429416,0.477244,0.075919
1,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.000000,0.012085,0.022114,0.058370,0.119054,0.049898


## 8. Text Baseline

This model uses the complaint narrative text. TF-IDF turns the text into numbers by giving more weight to words and phrases that are distinctive.

This section now uses two text views:
- word TF-IDF catches words and short phrases such as `air bag`, `caught fire`, or `brake failure`
- character TF-IDF catches smaller spelling patterns and helps when narratives contain typos or inconsistent wording

The vocabularies are capped so the notebook stays runnable on local hardware while still covering the strongest phrases and typo patterns.

In [8]:
# Keep the text settings moderate so the notebook stays runnable on local hardware while still surfacing strong narrative signals
WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 5)
TEXT_MIN_DF = 5
WORD_MAX_FEATURES = 20000
CHAR_MAX_FEATURES = 10000

train_text = train_df['cdescr_model_text']
valid_text = valid_df['cdescr_model_text']
holdout_text = holdout_df['cdescr_model_text']

text_preprocess = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=TEXT_MIN_DF,
        max_df=0.995,
        max_features=WORD_MAX_FEATURES,
        sublinear_tf=True,
        strip_accents='unicode',
        lowercase=True,
        dtype=np.float32
    )),
    ('char_tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=CHAR_NGRAM_RANGE,
        min_df=TEXT_MIN_DF,
        max_features=CHAR_MAX_FEATURES,
        sublinear_tf=True,
        strip_accents='unicode',
        lowercase=True,
        dtype=np.float32
    ))
])

X_train_text_matrix = text_preprocess.fit_transform(train_text)
X_valid_text_matrix = text_preprocess.transform(valid_text)
X_holdout_text_matrix = text_preprocess.transform(holdout_text)

text_model = build_linear_sgd_model()
text_model.fit(X_train_text_matrix, train_df[TARGET_COL])
text_valid_score = get_positive_scores(text_model, X_valid_text_matrix)
evaluate_scores('text_tfidf_word_char_sgd', 'valid_2025', valid_df[TARGET_COL].to_numpy(), text_valid_score)
results_df = show_results()

,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_1pct,recall_top_2pct,recall_top_5pct,recall_top_10pct,brier_score
0,text_tfidf_word_char_sgd,valid_2025,73985,0.052565,0.830288,0.963907,0.658561,0.534929,0.856518,0.189509,0.372846,0.750836,0.874518,0.040762
1,structured_core_sgd,valid_2025,73985,0.052565,0.444691,0.726783,0.344697,0.272388,0.469272,0.173052,0.342247,0.429416,0.477244,0.075919
2,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.000000,0.012085,0.022114,0.058370,0.119054,0.049898


## 9. Structured + Text Model

This section stacks the leakage-safe structured matrix and the fitted TF-IDF text matrix, then trains one more linear model on the combined sparse table.

Reusing the fitted transforms from sections 7 and 8 keeps the comparison fair and avoids vectorizing the full complaint corpus twice.

In [9]:
X_train_hybrid = sparse.hstack([
    X_train_structured_matrix,
    X_train_text_matrix
], format='csr')
X_valid_hybrid = sparse.hstack([
    X_valid_structured_matrix,
    X_valid_text_matrix
], format='csr')
X_holdout_hybrid = sparse.hstack([
    X_holdout_structured_matrix,
    X_holdout_text_matrix
], format='csr')

hybrid_model = build_linear_sgd_model()
hybrid_model.fit(X_train_hybrid, train_df[TARGET_COL])
hybrid_valid_score = get_positive_scores(hybrid_model, X_valid_hybrid)
evaluate_scores('hybrid_structured_text_sgd', 'valid_2025', valid_df[TARGET_COL].to_numpy(), hybrid_valid_score)

comparison_cols = [
    'model',
    'pr_auc',
    'precision_at_0_5',
    'recall_at_0_5',
    'recall_top_1pct',
    'recall_top_2pct',
    'recall_top_5pct',
    'recall_top_10pct',
    'brier_score'
]
comparison_df = pd.DataFrame(severity_results)
comparison_df = comparison_df.loc[
    comparison_df['split'].eq('valid_2025')
    & comparison_df['model'].isin([
        'dummy_prior',
        'structured_core_sgd',
        'text_tfidf_word_char_sgd',
        'hybrid_structured_text_sgd'
    ]),
    comparison_cols
].sort_values('pr_auc', ascending=False)

display(comparison_df)
results_df = show_results()

,model,pr_auc,precision_at_0_5,recall_at_0_5,recall_top_1pct,recall_top_2pct,recall_top_5pct,recall_top_10pct,brier_score
2,text_tfidf_word_char_sgd,0.830288,0.534929,0.856518,0.189509,0.372846,0.750836,0.874518,0.040762
3,hybrid_structured_text_sgd,0.821593,0.681078,0.806120,0.188995,0.372589,0.744150,0.869632,0.024998
1,structured_core_sgd,0.444691,0.272388,0.469272,0.173052,0.342247,0.429416,0.477244,0.075919
0,dummy_prior,0.052565,0.000000,0.000000,0.012085,0.022114,0.058370,0.119054,0.049898


,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_1pct,recall_top_2pct,recall_top_5pct,recall_top_10pct,brier_score
0,text_tfidf_word_char_sgd,valid_2025,73985,0.052565,0.830288,0.963907,0.658561,0.534929,0.856518,0.189509,0.372846,0.750836,0.874518,0.040762
1,hybrid_structured_text_sgd,valid_2025,73985,0.052565,0.821593,0.955484,0.738342,0.681078,0.806120,0.188995,0.372589,0.744150,0.869632,0.024998
2,structured_core_sgd,valid_2025,73985,0.052565,0.444691,0.726783,0.344697,0.272388,0.469272,0.173052,0.342247,0.429416,0.477244,0.075919
3,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.000000,0.012085,0.022114,0.058370,0.119054,0.049898


## 10. Review Budgets On The Validation Set

The table below converts each model's `valid_2025` scores into concrete review queues. This is the main decision surface for the urgency rule.

In [10]:
valid_score_lookup = {
    'dummy_prior': dummy_valid_score,
    'structured_core_sgd': structured_valid_score,
    'text_tfidf_word_char_sgd': text_valid_score,
    'hybrid_structured_text_sgd': hybrid_valid_score
}

valid_budget_tables = []
for model_name, score in valid_score_lookup.items():
    valid_budget_tables.append(
        build_operating_point_table(
            model_name,
            'valid_2025',
            valid_df[TARGET_COL].to_numpy(),
            score
        )
    )

valid_budget_df = pd.concat(valid_budget_tables, ignore_index=True)
valid_budget_df = valid_budget_df.sort_values(
    ['budget_fraction', 'recall_within_flagged_set', 'precision_within_flagged_set'],
    ascending=[True, False, False]
).reset_index(drop=True)

display(valid_budget_df[[
    'model',
    'budget_label',
    'flagged_rows',
    'flagged_share',
    'severe_cases_captured',
    'recall_within_flagged_set',
    'precision_within_flagged_set',
    'lift_vs_base_rate',
    'score_cutoff'
]])

,model,budget_label,flagged_rows,flagged_share,severe_cases_captured,recall_within_flagged_set,precision_within_flagged_set,lift_vs_base_rate,score_cutoff
0,text_tfidf_word_char_sgd,top_1pct,740,0.010002,737,0.189509,0.995946,18.947046,0.999629
1,hybrid_structured_text_sgd,top_1pct,740,0.010002,735,0.188995,0.993243,18.895629,1.000000
2,structured_core_sgd,top_1pct,740,0.010002,673,0.173052,0.909459,17.301712,0.999999
3,dummy_prior,top_1pct,740,0.010002,47,0.012085,0.063514,1.208292,0.062402
4,text_tfidf_word_char_sgd,top_2pct,1480,0.020004,1450,0.372846,0.979730,18.638546,0.995543
5,hybrid_structured_text_sgd,top_2pct,1480,0.020004,1449,0.372589,0.979054,18.625691,0.999987
6,structured_core_sgd,top_2pct,1480,0.020004,1331,0.342247,0.899324,17.108899,0.999986
7,dummy_prior,top_2pct,1480,0.020004,86,0.022114,0.058108,1.105459,0.062402
8,text_tfidf_word_char_sgd,top_5pct,3700,0.050010,2920,0.750836,0.789189,15.013670,0.790055
9,hybrid_structured_text_sgd,top_5pct,3700,0.050010,2894,0.744150,0.782162,14.879987,0.767988


## 11. Choose The Review Rule

Text-only is still the strongest pure ranker so far. Hybrid can still win the urgency-rule role if its top 5% queue is competitive enough and its scores look more usable for triage.

The selection rule in this notebook is:
- choose the model with the best `recall_top_5pct`
- if two models are within `0.01` recall at 5%, choose the one with better precision at 5%
- if still tied, choose higher PR-AUC
- if still tied, choose lower Brier score

The 1%, 2%, and 10% budget tiers stay visible as supporting checks.

In [11]:
selection_base = show_results(display_table=False)
selection_base = selection_base.loc[
    selection_base['split'].eq('valid_2025'),
    ['model', 'pr_auc', 'brier_score']
].set_index('model')

selection_parts = [selection_base]
for metric_name, prefix in [
    ('recall_within_flagged_set', 'recall'),
    ('precision_within_flagged_set', 'precision'),
    ('lift_vs_base_rate', 'lift')
]:
    metric_frame = valid_budget_df.pivot(index='model', columns='budget_label', values=metric_name)
    metric_frame.columns = [f'{prefix}_{col}' for col in metric_frame.columns]
    selection_parts.append(metric_frame)

selection_df = selection_parts[0]
for part in selection_parts[1:]:
    selection_df = selection_df.join(part)

selection_df = selection_df.reset_index()[[
    'model',
    'recall_top_1pct',
    'precision_top_1pct',
    'recall_top_2pct',
    'precision_top_2pct',
    'recall_top_5pct',
    'precision_top_5pct',
    'recall_top_10pct',
    'precision_top_10pct',
    'pr_auc',
    'brier_score',
    'lift_top_5pct'
]]

selection_df, selected_model_row = choose_urgency_model(selection_df)
selected_model_name = selected_model_row['model']

print('Selected urgency rule:', selected_model_name)
print('Primary review budget:', 'top 5%')
print('Top 5% recall:', round(float(selected_model_row['recall_top_5pct']), 4))
print('Top 5% precision:', round(float(selected_model_row['precision_top_5pct']), 4))
print('Validation PR-AUC:', round(float(selected_model_row['pr_auc']), 4))
print('Validation Brier score:', round(float(selected_model_row['brier_score']), 4))

display(selection_df)

Selected urgency rule: text_tfidf_word_char_sgd
Primary review budget: top 5%
Top 5% recall: 0.7508
Top 5% precision: 0.7892
Validation PR-AUC: 0.8303
Validation Brier score: 0.0408


,model,recall_top_1pct,precision_top_1pct,recall_top_2pct,precision_top_2pct,recall_top_5pct,precision_top_5pct,recall_top_10pct,precision_top_10pct,pr_auc,brier_score,lift_top_5pct,within_0_01_recall_of_best_top_5pct
0,text_tfidf_word_char_sgd,0.189509,0.995946,0.372846,0.979730,0.750836,0.789189,0.874518,0.459657,0.830288,0.040762,15.013670,True
1,hybrid_structured_text_sgd,0.188995,0.993243,0.372589,0.979054,0.744150,0.782162,0.869632,0.457089,0.821593,0.024998,14.879987,True
2,structured_core_sgd,0.173052,0.909459,0.342247,0.899324,0.429416,0.451351,0.477244,0.250845,0.444691,0.075919,8.586585,False
3,dummy_prior,0.012085,0.063514,0.022114,0.058108,0.058370,0.061351,0.119054,0.062576,0.052565,0.049898,1.167159,False


## 12. Review The Chosen Queue

This section turns the selected `valid_2025` score into notebook urgency bands and shows the kinds of cases that would rise to reviewers first.

It also surfaces suspicious-value flags in the review tables as quality context for the reviewer rather than using them as model features.

In [12]:
selected_valid_df = valid_df.copy()
suspicious_flag_cols = [
    col for col in [
        'flag_speed_999',
        'flag_speed_high',
        'flag_miles_high',
        'flag_injured_99',
        'flag_deaths_99'
    ]
    if col in selected_valid_df.columns
]
for col in suspicious_flag_cols:
    selected_valid_df[col] = selected_valid_df[col].fillna(False).astype(bool)
selected_valid_df['suspicious_value_any'] = (
    selected_valid_df[suspicious_flag_cols].any(axis=1)
    if suspicious_flag_cols
    else False
)
selected_valid_df['urgency_score'] = valid_score_lookup[selected_model_name]
selected_valid_df['urgency_band'] = assign_urgency_bands(selected_valid_df['urgency_score'])
selected_valid_df['urgency_band'] = pd.Categorical(
    selected_valid_df['urgency_band'],
    categories=['high_urgency', 'priority_review', 'watch', 'routine'],
    ordered=True
)
selected_valid_df['review_month'] = selected_valid_df['ldate'].dt.to_period('M').astype(str)

band_summary = selected_valid_df['urgency_band'].value_counts(sort=False).rename_axis('urgency_band').reset_index(name='rows')
band_summary['share'] = band_summary['rows'] / len(selected_valid_df)

monthly_review_burden = selected_valid_df.groupby(['review_month', 'urgency_band'], observed=False).size().unstack(fill_value=0).reset_index()

review_cols = [
    'odino',
    'ldate',
    'mfr_name',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'urgency_score',
    'urgency_band',
    'suspicious_value_any',
    'severity_primary_flag',
    'severity_broad_flag',
    'cdescr_model_text'
]
review_cols = review_cols[:-1] + suspicious_flag_cols + review_cols[-1:]

top_risk_df = selected_valid_df.sort_values('urgency_score', ascending=False)[review_cols].head(25)
false_positive_df = selected_valid_df.loc[
    selected_valid_df['urgency_band'].eq('high_urgency')
    & ~selected_valid_df[TARGET_COL],
    review_cols
].sort_values('urgency_score', ascending=False).head(25)
missed_severe_df = selected_valid_df.loc[
    selected_valid_df[TARGET_COL]
    & selected_valid_df['urgency_band'].eq('routine'),
    review_cols
].sort_values('urgency_score', ascending=False).head(25)

print('Urgency band counts on valid_2025')
display(band_summary)

print('Monthly review burden on valid_2025')
display(monthly_review_burden)

print('Top 25 highest-scored complaints on valid_2025')
display(top_risk_df)

print('Top-band false positives on valid_2025')
display(false_positive_df)

print('Missed severe complaints outside the top 10% on valid_2025')
display(missed_severe_df)

Urgency band counts on valid_2025


,urgency_band,rows,share
0,high_urgency,739,0.009989
1,priority_review,2960,0.040008
2,watch,3699,0.049997
3,routine,66587,0.900007


Monthly review burden on valid_2025


urgency_band,review_month,high_urgency,priority_review,watch,routine
0,2025-01,54,231,288,5911
1,2025-02,48,197,256,5091
2,2025-03,54,211,277,5630
3,2025-04,49,177,269,5577
4,2025-05,56,221,282,5458
5,2025-06,53,218,281,5658
6,2025-07,76,279,354,6524
7,2025-08,73,282,335,5808
8,2025-09,71,310,339,5639
9,2025-10,73,275,354,5451


Top 25 highest-scored complaints on valid_2025


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,urgency_score,urgency_band,suspicious_value_any,severity_primary_flag,severity_broad_flag,flag_speed_999,flag_speed_high,flag_miles_high,flag_injured_99,flag_deaths_99,cdescr_model_text
295452,11638081,2025-01-23,"BMW OF NORTH AMERICA, LLC",MINI,COOPER,2009,1.000000,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2009 Mini Cooper. The cont...
361453,11704694,2025-12-12,"SUBARU OF AMERICA, INC.",SUBARU,FORESTER,2019,1.000000,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2019 Subaru Forester. The ...
352572,11695774,2025-10-27,"TESLA, INC.",TESLA,MODEL Y,2026,1.000000,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2026 Tesla Model Y. The co...
342897,11685973,2025-09-09,"CHRYSLER (FCA US, LLC)",RAM,1500,2021,1.000000,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2021 RAM 1500. The contact...
347888,11691052,2025-10-02,FORD MOTOR COMPANY,FORD,BRONCO SPORT,2023,0.999999,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2023 Ford Bronco Sport. Th...
351562,11694758,2025-10-21,TOYOTA MOTOR CORPORATION,TOYOTA,RAV4,2021,0.999999,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2021 Toyota Rav4. The cont...
337150,11680186,2025-08-12,"VOLKSWAGEN GROUP OF AMERICA, INC.",AUDI,A4 ALLROAD,2013,0.999999,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2013 Audi A4 Allroad. The ...
302420,11645124,2025-02-26,"CHRYSLER (FCA US, LLC)",JEEP,GRAND CHEROKEE 4XE,2023,0.999999,high_urgency,False,True,True,False,False,False,False,False,The contact rented a 2023 Jeep Grand Cherokee....
363362,11706615,2025-12-22,"GENERAL MOTORS, LLC",BUICK,ENCORE,2019,0.999999,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2019 Buick Encore. The con...
310872,11653688,2025-04-09,"CHRYSLER (FCA US, LLC)",RAM,PROMASTER,2021,0.999999,high_urgency,False,True,True,False,False,False,False,False,The contact owned a 2021 Ram 3500 Promaster. T...


Top-band false positives on valid_2025


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,urgency_score,urgency_band,suspicious_value_any,severity_primary_flag,severity_broad_flag,flag_speed_999,flag_speed_high,flag_miles_high,flag_injured_99,flag_deaths_99,cdescr_model_text
343980,11687059,2025-09-13,FORD MOTOR COMPANY,LINCOLN,MKC,2017,0.999931,high_urgency,False,False,False,False,False,False,False,False,None of my airbags deployed in a one vehicle a...
338360,11681405,2025-08-18,TOYOTA MOTOR CORPORATION,TOYOTA,CAMRY,2022,0.999871,high_urgency,False,False,False,False,False,False,False,False,Heated steering wheel so hot on the left side....
328148,11671131,2025-07-03,"CHRYSLER (FCA US, LLC)",CHRYSLER,TOWN AND COUNTRY,2014,0.999744,high_urgency,False,False,False,False,False,False,False,False,The contact's partner owns a 2014 Chrysler Tow...


Missed severe complaints outside the top 10% on valid_2025


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,urgency_score,urgency_band,suspicious_value_any,severity_primary_flag,severity_broad_flag,flag_speed_999,flag_speed_high,flag_miles_high,flag_injured_99,flag_deaths_99,cdescr_model_text
351035,11694223,2025-10-18,FORD MOTOR COMPANY,FORD,ESCAPE,2014,0.415895,routine,False,True,True,False,False,False,False,False,see attached documents: Subject: Ongoing Unres...
346237,11689392,2025-09-24,"INEOS AUTOMOTIVE AMERICAS, LLC",INEOS,GRENADIER,2024,0.414065,routine,False,True,True,False,False,False,False,False,The contact owns a 2024 Ineos Grenadier. The c...
348405,11691578,2025-10-06,"GENERAL MOTORS, LLC",CHEVROLET,BLAZER EV,2024,0.412779,routine,False,True,True,False,False,False,False,False,My son was driving in stop and go traffic on I...
329977,11672963,2025-07-11,TOYOTA MOTOR CORPORATION,LEXUS,IS,2009,0.411387,routine,False,True,True,False,False,False,False,False,"The dashboard is melting leather, as well as t..."
301256,11643951,2025-02-20,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2017,0.409685,routine,False,True,True,False,False,False,False,False,Electrical System Failure Battery Drain Issues...
356334,11699556,2025-11-15,"TESLA, INC.",TESLA,MODEL Y,2023,0.409296,routine,False,True,True,False,False,False,False,False,The car don't run well
293059,11635666,2025-01-13,FISKER GROUP INC,FISKER,OCEAN,2023,0.408707,routine,False,True,True,False,False,False,False,False,I was scheduled to receive the 2.2 software up...
330804,11673804,2025-07-15,"VOLKSWAGEN GROUP OF AMERICA, INC.",VOLKSWAGEN,JETTA,2011,0.407863,routine,False,True,True,False,False,False,False,False,driving on freeway (speed around 70mph) out of...
360780,11704018,2025-12-09,"GENERAL MOTORS, LLC",CHEVROLET,BLAZER EV,2025,0.407552,routine,False,True,True,False,False,False,False,False,"At a low speed, between 10 and 15 mph, the for..."
346444,11689600,2025-09-25,"GENERAL MOTORS, LLC",CHEVROLET,IMPALA,2010,0.407110,routine,False,True,True,False,False,False,False,False,I bought a used car as is for $6700.00 and was...


## 13. Calibration Check

If the score is only used to sort a queue, calibration is secondary. If the score is used like an urgency probability, calibration matters more.

The simple bin tables below keep that distinction visible without adding heavy plotting or pipeline logic.

In [13]:
calibration_models = {
    'text_tfidf_word_char_sgd': text_valid_score,
    'hybrid_structured_text_sgd': hybrid_valid_score
}

for model_name, score in calibration_models.items():
    print(model_name)
    display(build_calibration_table(valid_df[TARGET_COL].to_numpy(), score, n_bins=10))

text_tfidf_word_char_sgd


,bin,rows,avg_score,observed_rate,min_score,max_score,gap
0,"(0.999, 7399.4]",7399,0.006334,0.000811,0.000701,0.010082,-0.005523
1,"(7399.4, 14797.8]",7398,0.013923,0.001487,0.010082,0.017929,-0.012436
2,"(14797.8, 22196.2]",7399,0.022522,0.001352,0.017930,0.027551,-0.021170
3,"(22196.2, 29594.6]",7398,0.033631,0.002568,0.027552,0.040329,-0.031063
4,"(29594.6, 36993.0]",7399,0.048401,0.003649,0.040333,0.057329,-0.044752
5,"(36993.0, 44391.4]",7398,0.069209,0.004731,0.057332,0.082539,-0.064478
6,"(44391.4, 51789.8]",7398,0.100679,0.008110,0.082544,0.122621,-0.092569
7,"(51789.8, 59188.2]",7399,0.156442,0.012029,0.122623,0.200715,-0.144413
8,"(59188.2, 66586.6]",7398,0.284127,0.031225,0.200721,0.416804,-0.252903
9,"(66586.6, 73985.0]",7399,0.764065,0.459657,0.416814,1.000000,-0.304408


hybrid_structured_text_sgd


,bin,rows,avg_score,observed_rate,min_score,max_score,gap
0,"(0.999, 7399.4]",7399,0.000072,0.003379,1.719032e-18,0.000158,0.003307
1,"(7399.4, 14797.8]",7398,0.000277,0.002163,1.577138e-04,0.000420,0.001886
2,"(14797.8, 22196.2]",7399,0.000632,0.002973,4.199302e-04,0.000881,0.002341
3,"(22196.2, 29594.6]",7398,0.001240,0.002703,8.810836e-04,0.001663,0.001464
4,"(29594.6, 36993.0]",7399,0.002302,0.004866,1.663294e-03,0.003084,0.002563
5,"(36993.0, 44391.4]",7398,0.004350,0.005137,3.084754e-03,0.006003,0.000787
6,"(44391.4, 51789.8]",7398,0.008798,0.006218,6.004709e-03,0.012596,-0.002580
7,"(51789.8, 59188.2]",7399,0.020598,0.012299,1.259648e-02,0.032766,-0.008299
8,"(59188.2, 66586.6]",7398,0.073010,0.028792,3.276970e-02,0.159402,-0.044218
9,"(66586.6, 73985.0]",7399,0.671551,0.457089,1.594121e-01,1.000000,-0.214463


## 14. Optional: Reference Holdout Check

`holdout_2026` is now reference-only. Leave this off during notebook development and only run it once you intentionally want a single future-period readout for the rule chosen on `valid_2025`.

In [14]:
RUN_HOLDOUT = False
BEST_MODEL_NAME = selected_model_name

if RUN_HOLDOUT:
    model_lookup = {
        'dummy_prior': dummy_model,
        'structured_core_sgd': structured_model,
        'text_tfidf_word_char_sgd': text_model,
        'hybrid_structured_text_sgd': hybrid_model
    }
    input_lookup = {
        'dummy_prior': holdout_df[['odino']],
        'structured_core_sgd': X_holdout_structured_matrix,
        'text_tfidf_word_char_sgd': X_holdout_text_matrix,
        'hybrid_structured_text_sgd': X_holdout_hybrid
    }
    if BEST_MODEL_NAME not in model_lookup:
        raise ValueError(f'BEST_MODEL_NAME must be one of {sorted(model_lookup)}')

    holdout_score = get_positive_scores(model_lookup[BEST_MODEL_NAME], input_lookup[BEST_MODEL_NAME])
    evaluate_scores(BEST_MODEL_NAME, 'holdout_2026', holdout_df[TARGET_COL].to_numpy(), holdout_score)
    results_df = show_results()
else:
    print('Holdout skipped. Leave RUN_HOLDOUT = False until you want one reference-only 2026 readout for the chosen valid_2025 urgency rule.')

Holdout skipped. Leave RUN_HOLDOUT = False until you want one reference-only 2026 readout for the chosen valid_2025 urgency rule.


## 15. Save Results

This cell writes the compact exploratory results summary used for sharing notebook benchmark results.

In [15]:
results_df = show_results(display_table=False)
results_df.to_csv(SEVERITY_RESULTS_PATH, index=False)
print('Wrote:', SEVERITY_RESULTS_PATH)

Wrote: C:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\outputs\severity_partner_results.csv


## 16. Next Notebook Moves

Use this section to guide the next exploration pass after the urgency-rule framing is in place.

Suggested next steps:
- Review the top-band false positives and missed severe cases for repeated patterns that suggest better text cleaning or feature design
- Add leakage-safe cohort-history features and check whether they improve the top 5% review queue without overcomplicating the notebook
- Run a later sensitivity pass with `severity_broad_flag` once the primary urgency rule feels stable
- If the chosen score needs to behave more like a probability, calibrate only the shortlisted model instead of broadening the whole benchmark set
- Run the optional 2026 reference check once after the `valid_2025` rule and review bands are locked